In [10]:
# import pandas as pd
import requests
import json
import re

START_YEAR = 2010
END_YEAR = 2020

# id is just the row number PER YEAR, but not super meaningful.
# it is included in the JSON data.
COLUMNS = ['id', 'Year', 'Location', 'First Name', 'Last Name', 'Title', 'Gross Pay', 'Regular Pay', 'Overtime Pay', 'Other Pay']

def safe_parse(value):
    try:
        return float(value)
    except:
        return value

def load_data_for_year(year):
    raw_data = open(f"all-records-{year}.json").read()
    raw_data = raw_data.replace("'", '"')
    raw_data = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', raw_data)
    parsed = json.loads(raw_data)
    data = [ [ safe_parse(value) for value in row['cell'] ] for row in parsed['rows'] ]
    return pd.DataFrame(data, columns= COLUMNS)

# for i in range(START_YEAR, END_YEAR + 1):
#     df = load_data_for_year(i)
#     df.to_csv(f'all-records-{i}.csv')

In [3]:
%pip install pandas
%pip install requests

  Using cached pandas-2.2.3-cp312-cp312-macosx_10_9_x86_64.whl.metadata (89 kB)
  Using cached numpy-2.2.5-cp312-cp312-macosx_14_0_x86_64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.2.3-cp312-cp312-macosx_10_9_x86_64.whl (12.5 MB)
Using cached numpy-2.2.5-cp312-cp312-macosx_14_0_x86_64.whl (6.7 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]
Note: you may need to restart the kernel to use updated packages.
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached charset_normalizer-3.4.1-cp312-cp312-macosx_10_13_universal2.whl.metadata (35 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
Using cached requests-2.32.3-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.1-cp312

# Download Data

```sh
curl 'https://ucannualwage.ucop.edu/wage/search.do?_search=false&rows=9999&page=1&sidx=EAW_LST_NAM&sord=asc&year=2023&location=Los+Angeles&firstname=&lastname=&title=&startSal=&endSal='
```

The UCAnnualwage site typically uses a POST, but the cURL command as a GET works easily enough.
It does not seem to limit the value of the `rows` parameter, so you can set it to a large number to get all the data in one request.
However the server seems to return an OOM exception if you set it too high.

Downloading all data must depaginate the results.

The JSON data looks like this:

```json
{
'page' : '1',
'total' : '0',
'records' : '305565',
'rows':[
{'id':'1','cell':['1','2021','','','','','0.00','0.00','0.00','0.00']},
...
]
}
```

# All Campuses



In [ ]:
CAMPUS = (
"ALL", # Not acceptable in queries which do not have other filters than the campus parameter
"Berkeley",
# "DANR","DANR" # Not used since 2013
"Davis",
"Irvine",
"Los Angeles",
"Merced",
"Riverside",
"San Diego",
"San Francisco",
"Santa Barbara",
"Santa Cruz",
"UCOP" )

In [ ]:
UC_ANNUAL_WAGE_URL = "https://ucannualwage.ucop.edu/wage/search.do?"
QUERY = 'rows={rows}&page={page}&sidx=EAW_LST_NAM&sord=asc&year={year}&location={location}'
_unsed_params = '_search=false&firstname=&lastname=&title=&startSal=&endSal='
def query_campus_year(year, campus="ALL"):
    # Just the `rows` value of the query response.
    json_results = []
    page = 1
    total_pages = int(1e6)
    # As of now the API does not actually seem to limit the number of rows returned
    # but the server will OOM if you ask for too many rows....
    rows = 50000
    location = campus.replace(" ", "+")
    complete_url = UC_ANNUAL_WAGE_URL + QUERY.format(rows=rows, page=page, year=year, location=location)
    expected_rows = 0
    while True:
        print(f"Page {page} - Querying {complete_url}")
        response = requests.get(complete_url)
        if response.status_code != 200:
            print(f"Error: {response.status_code} for {complete_url}")
            break
        print(f"\t Got {len(response.content)} bytes")
        data = response.json()
        if not expected_rows:
            expected_rows = int(data.get('records', 0))
            print(f"\t Expected {expected_rows} rows")
        print(f"\t Got {len(data.get('rows', []))} rows")
        json_results += data.get('rows', [])
        if len(json_results) >= expected_rows:
            break
        total_pages = int(data.get('total', 0))
        if page >= total_pages:
            break
        page += 1
        complete_url = UC_ANNUAL_WAGE_URL + QUERY.format(rows=rows, page=page, year=year, location=location)
    return json_results

In [ ]:
results_2022 = query_campus_year(2022)
# write to json
with open('data/json/all-records-2022.json', 'w') as f:
    json.dump(results_2022, f, indent=2)
print(f"Got {len(results_2022)} records for 2022")

# write to csv
# df_2023 = pd.DataFrame.from_dict(results_2023, orient='index')
# df_2023.columns = COLUMNS
# df_2023.to_csv('uc_2023.csv', index=False)

Page 1 - Querying https://ucannualwage.ucop.edu/wage/search.do?rows=50000&page=1&sidx=EAW_LST_NAM&sord=asc&year=2022&location=ALL
